# Benchmark Localization Map Crossmatch

- [ ] Download ~30 mocs (for now)
- [ ] Get some df with sample or fake data with radecs (can be just ~100 or 1k rows, for now)
    - [ ] see notes from [[2026-06-17]] on using lsdb data generation for this
- [ ] Query whether or not each row is in each moc
- [ ] Once notebook is working, scale up (both num mocs and num rows)
    - [ ] Maybe switch to using real data

In [1]:
import warnings

warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")

## DL the MOCs
- For now just going with however many they give (this seems to be about 14),
- but we'll want to scale this up (we'd talked about ~300)
- The number we get rn seems determined by our selection criteria
  - BNS + NSBH are giving ~14 (and these are the ones we'd want to respond to rapidly)
  - BBH would be exected to be far more

In [2]:
from desi_aap import load_config
from desi_aap.gracedb_tools import fetch_gracedb_superevents

cfg = load_config("../../config.toml")
gracedb_events = fetch_gracedb_superevents(se_types=["NSBH"], cache=cfg.gracedb.to_cache())
gracedb_events

,superevent_id,gw_time,gps_time,far_hz,far_per_year,p_bns,p_nsbh,p_bbh,p_terrestrial,classification_file,preferred_event,pipeline,search,instruments,labels,skymap_file,skymap_path,status,cache_status
0,S190814bv,2019-08-14 21:10:39.012957+00:00,1.249852e+09,2.032625e-33,6.414477e-26,0.000000e+00,0.997890,0.000000e+00,0.000000e+00,p_astro.json,G347305,gstlal,AllSky,"H1,L1,V1","PE_READY,ADVOK,SKYMAP_READY,EMBRIGHT_READY,PAS...",bayestar.multiorder.fits,/Users/scampos/desi-alert-augmentation-pipelin...,ok,miss
1,S190910d,2019-09-10 01:26:19.242676+00:00,1.252114e+09,3.717180e-09,1.173053e-01,0.000000e+00,0.975899,0.000000e+00,2.410074e-02,p_astro.json,G350002,spiir,HighMass,"H1,L1","PE_READY,ADVOK,SKYMAP_READY,EMBRIGHT_READY,PAS...",bayestar.multiorder.fits,/Users/scampos/desi-alert-augmentation-pipelin...,ok,miss
2,S191117j,2019-11-17 06:08:22.454868+00:00,1.258006e+09,1.114482e-18,3.517037e-11,0.000000e+00,1.000000,0.000000e+00,1.080973e-10,p_astro.json,G354833,gstlal,AllSky,"H1,L1","ADVNO,EM_Selected,SKYMAP_READY,EMBRIGHT_READY,...",bayestar.multiorder.fits,/Users/scampos/desi-alert-augmentation-pipelin...,ok,miss
3,S191205ah,2019-12-05 21:52:08.568738+00:00,1.259618e+09,1.248393e-08,3.939628e-01,0.000000e+00,0.932102,0.000000e+00,6.789774e-02,p_astro.json,G356642,gstlal,AllSky,"H1,L1,V1","EM_READY,ADVOK,EM_Selected,SKYMAP_READY,EMBRIG...",bayestar.multiorder.fits,/Users/scampos/desi-alert-augmentation-pipelin...,ok,miss
4,S200116ah,2020-01-16 11:56:42.170712+00:00,1.263211e+09,2.028966e-12,6.402930e-05,0.000000e+00,0.999934,0.000000e+00,6.579270e-05,p_astro.json,G360499,gstlal,AllSky,"H1,L1","EM_READY,PE_READY,ADVNO,EM_Selected,SKYMAP_REA...",bayestar.multiorder.fits,/Users/scampos/desi-alert-augmentation-pipelin...,ok,miss
5,S230715bw,2023-07-15 19:05:37.952637+00:00,1.373483e+09,7.843396e-09,2.475188e-01,0.000000e+00,0.906988,8.832113e-02,4.690587e-03,spiir.p_astro.json,G417598,spiir,AllSky,"H1,L1","EM_READY,PE_READY,ADVNO,SKYMAP_READY,EMBRIGHT_...",bayestar.multiorder.fits,/Users/scampos/desi-alert-augmentation-pipelin...,ok,miss
6,S240422ed,2024-04-22 21:35:13.417261+00:00,1.397857e+09,3.095258e-13,9.767891e-06,1.035171e-16,0.999986,7.235380e-16,1.384899e-05,gstlal.p_astro.json,G476954,gstlal,AllSky,"H1,L1,V1","EM_READY,PE_READY,ADVOK,SKYMAP_READY,EMBRIGHT_...",bayestar.multiorder.fits,/Users/scampos/desi-alert-augmentation-pipelin...,ok,miss


## Generate fake data
- For now, can be just ~100 or 1k rows
- then we'll scale up (expect ~10k per night)
- use the lsdb.nested.generate code for this

In [3]:
import numpy as np
import pandas as pd
from lsdb import ConeSearch
from lsdb import generate_data


# Generate fake data
ddf = generate_data(10_000, 0, search_region=ConeSearch(288, 48, radius_arcsec=11 * 3600))
df = ddf.compute()

# Remove the col labeled "nested"
df = df.drop(columns=["nested"])

# Add a col labeled "discoverydate" and give it random timestamps from the past year
now = pd.Timestamp.now(tz="utc")
random_offsets = pd.to_timedelta(np.random.uniform(0, 365, size=df.shape[0]), unit="D")
df["discoverydate"] = now - random_offsets

df

,ra,dec,id,a,b,discoverydate
0,299.993695,53.831457,29429,0.857678,1.066060,2026-01-12 17:22:04.112796686+00:00
1,276.406461,55.658529,13205,0.733983,0.544663,2025-09-01 08:04:45.594325569+00:00
...,...,...,...,...,...,...
9998,283.218577,39.878061,29571,0.747701,0.675289,2025-12-29 22:24:03.710497841+00:00
9999,282.595814,44.007119,58752,0.990646,1.879456,2025-10-02 08:07:46.720934219+00:00


In [4]:
# Add cols labeled "dist_mpc_SHOES", "dist_mpc_Planck18" with randomly generated vals (0.0, 500.0)

df["dist_mpc_SHOES"] = np.random.uniform(0.0, 500.0, size=len(df))
df["dist_mpc_Planck18"] = np.random.uniform(0.0, 500.0, size=len(df))

df

,ra,dec,id,a,b,discoverydate,dist_mpc_SHOES,dist_mpc_Planck18
0,299.993695,53.831457,29429,0.857678,1.066060,2026-01-12 17:22:04.112796686+00:00,60.672674,65.016318
1,276.406461,55.658529,13205,0.733983,0.544663,2025-09-01 08:04:45.594325569+00:00,187.641702,202.878517
...,...,...,...,...,...,...,...,...
9998,283.218577,39.878061,29571,0.747701,0.675289,2025-12-29 22:24:03.710497841+00:00,266.715719,113.395894
9999,282.595814,44.007119,58752,0.990646,1.879456,2025-10-02 08:07:46.720934219+00:00,224.334511,438.168530


In [5]:
# Add a declination col that's a copy of the dec column
df["declination"] = df["dec"]

## Crossmatch: is each row in each MOC?

- This will be the main thing to change, if this turns out to be too slow. Ideally the ligo code is fast enough; could be we need to get creative about performing this crossmatch.
- Note there are two "classes" of GW: the BBH events (which we'll have more of over the past year) and the NSBH+BNS events (which will be important to act fast on)
  - In theory, we could plan two phases of the crossmatch: first to the smaller group of more-urgent events, the NSBH+BNS events
  - Then run the crossmatch against the ~300 BBH events from the past year
  - Though, if we won't be sending the enhanced alert back right after the first phase, maybe it makes no difference...
- We'll want to add data to the enhanced alert:
  - Every GW it matched with
  - The specific contour region of each of those GWs

### Temporal, 2D, and 3D: all the facets of the crossmatch

1. First pass: a temporal crossmatch
   - The event can only match with a GW with a temporal window that contains its discovery date
2. Second pass: spatial (when calling LIGO code, this happens at the same time)
   - 2D: "the sky-only credible level after marginalizing over distance"
   - 3D: "a 3D voxel credible level ranked by posterior density per volume at (ra, dec, dist)"

In [6]:
from desi_aap.gracedb_tools import temporal_crossmatch_sesn_to_gw

temporal_xmatch = temporal_crossmatch_sesn_to_gw(df, gracedb_events)
temporal_xmatch

""


In [7]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.time import Time
from ligo.gracedb.rest import GraceDb
from ligo.skymap.io import read_sky_map
from ligo.skymap.postprocess import crossmatch

from desi_aap.cosmology import COSMOLOGIES


def failed_spatial_rows(sn_rows, status, cosmology=np.nan):
    """Mark SN rows as failing the spatial crossmatch with a given status."""
    out = sn_rows.copy()
    out["spatial_status"] = status
    out["cosmology"] = cosmology
    out["inside_2d_credible_level"] = False
    out["inside_3d_credible_level"] = False
    return out


def add_crossmatch_columns(sn_rows, result, cosmology_label, distance_column):
    """Attach ligo.skymap crossmatch result fields to a copy of the matched SN rows."""
    out = sn_rows.copy().reset_index(drop=True)
    out["cosmology"] = cosmology_label
    out["distance_column"] = distance_column
    out["sn_dist_mpc"] = out[distance_column]
    out["searched_area_deg2"] = np.atleast_1d(result.searched_area)
    out["searched_prob_2d"] = np.atleast_1d(result.searched_prob)
    out["offset_deg"] = np.atleast_1d(result.offset)
    out["searched_prob_dist"] = np.atleast_1d(result.searched_prob_dist)
    out["searched_vol_mpc3"] = np.atleast_1d(result.searched_vol)
    out["searched_prob_vol"] = np.atleast_1d(result.searched_prob_vol)
    out["searched_prob_3d_density_rank"] = out["searched_prob_vol"]
    out["probdensity_vol"] = np.atleast_1d(result.probdensity_vol)
    out["credible_volume_mpc3"] = result.contour_vols[0] if result.contour_vols else np.nan
    out["credible_area_deg2"] = result.contour_areas[0] if result.contour_areas else np.nan
    out["inside_2d_credible_level"] = out["searched_prob_2d"] <= CREDIBLE_LEVEL
    out["inside_3d_credible_level"] = out["searched_prob_vol"] <= CREDIBLE_LEVEL
    return out


# GraceDB settings.
GRACEDB_SERVICE_URL = "https://gracedb.ligo.org/api/"
GRACEDB_CATEGORY = "Production"
GRACEDB_MAX_RESULTS = None
GRACEDB_QUERY_SIGFIGS = 12

# Event-selection settings.
FAR_THRESHOLD_PER_YEAR = 2.0
JULIAN_YEAR_DAYS = 365.25
SECONDS_PER_DAY = (1 * u.day).to_value(u.s)
JULIAN_YEAR_SECONDS = (JULIAN_YEAR_DAYS * u.day).to_value(u.s)
FAR_THRESHOLD_HZ = FAR_THRESHOLD_PER_YEAR / JULIAN_YEAR_SECONDS
GRACEDB_QUERY = f"category: {GRACEDB_CATEGORY} far < {FAR_THRESHOLD_HZ:.{GRACEDB_QUERY_SIGFIGS}g}"
MIN_BNS_NSBH_PROB_SUM = 0.9  # TODO ask about mins for things like just BBH selection
DEFAULT_CLASSIFICATION_PROBABILITY = 0.0


# Temporal/spatial crossmatch settings.
TEMPORAL_WINDOW_DAYS = 14
CREDIBLE_LEVEL = 0.50
REQUIRE_2D_CREDIBLE_LEVEL = False

# crossmatch(..., cosmology=False) ranks the 3D posterior by probability
# density per luminosity-distance volume, matching the units in the skymaps.
# The two cosmology runs differ by the redshift-to-luminosity-distance
# conversion used for each SN.
USE_COMOVING_VOLUME_RANKING = True

# Local output directory for downloaded skymaps.
SKYMAP_DIR = Path("gracedb_skymaps")

# GraceDB skymap file-selection priorities. Lower is preferred.
SKYMAP_PRIORITY_BILBY_MULTIORDER = 0
SKYMAP_PRIORITY_BAYESTAR_MULTIORDER = 10
SKYMAP_PRIORITY_ANY_MULTIORDER = 20
SKYMAP_PRIORITY_BAYESTAR_FITS_GZ = 30
SKYMAP_PRIORITY_ANY_FITS_GZ = 40
SKYMAP_PRIORITY_ANY_FITS = 50
SKYMAP_VERSIONED_FILE_PRIORITY_PENALTY = 100
SKYMAP_PRIORITY_IGNORE = 1000


def run_3d_spatial_crossmatch(temporal_matches, gw_events):
    """Run the 3D credible-volume crossmatch for each cosmology on every temporal match."""
    if temporal_matches.empty or gw_events.empty:
        return pd.DataFrame()

    event_lookup = gw_events.set_index("superevent_id", drop=False)
    chunks = []
    skymap_cache = {}

    for superevent_id, sn_rows in temporal_matches.groupby("superevent_id"):
        # Get the event.
        if superevent_id not in event_lookup.index:
            continue
        event = event_lookup.loc[superevent_id]

        # Get the skymap (from cache, or get and cache it).
        skymap_path = event.get("skymap_path")
        if not skymap_path or not Path(skymap_path).exists():
            chunks.append(failed_spatial_rows(sn_rows, "missing_skymap"))
            continue
        if skymap_path not in skymap_cache:
            try:
                skymap_cache[skymap_path] = read_sky_map(skymap_path, moc=True)
            except Exception as exc:
                chunks.append(failed_spatial_rows(sn_rows, f"skymap_read_failed: {exc}"))
                continue
        skymap = skymap_cache[skymap_path]

        # Get the distance, available.
        if "DISTMU" not in skymap.colnames:
            chunks.append(failed_spatial_rows(sn_rows, "skymap_has_no_distance_columns"))
            continue

        # Calculate distance and crossmatch for both SHOES and Planck18.
        for cosmology_label in COSMOLOGIES:
            # Get distance wrt specific cosmology model.
            distance_column = f"dist_mpc_{cosmology_label}"
            valid = sn_rows[np.isfinite(sn_rows[distance_column])].copy()
            if valid.empty:
                continue
            coords = SkyCoord(
                ra=valid["ra"].to_numpy() * u.deg,
                dec=valid["declination"].to_numpy() * u.deg,
                distance=valid[distance_column].to_numpy() * u.Mpc,
                frame="icrs",
            )

            # Run crossmatch for that distance.
            try:
                # TODO: what format/schema/etc is result? and what does out look like?
                result = crossmatch(
                    skymap,
                    coords,
                    contours=(CREDIBLE_LEVEL,),
                    cosmology=USE_COMOVING_VOLUME_RANKING,
                )
                out = add_crossmatch_columns(valid, result, cosmology_label, distance_column)
                out["spatial_status"] = "ok"
            except Exception as exc:
                out = failed_spatial_rows(valid, f"crossmatch_failed: {exc}", cosmology_label)
                out["distance_column"] = distance_column
            chunks.append(out)

    # Return a sorted dataframe.
    if not chunks:
        return pd.DataFrame()
    df = pd.concat(chunks, ignore_index=True)
    sort_cols = [c for c in ["superevent_id", "name", "cosmology"] if c in df.columns]
    if sort_cols:
        df = df.sort_values(sort_cols)
    return df.reset_index(drop=True)

In [8]:
from desi_aap.gracedb_tools import run_3d_spatial_crossmatch

# TODO note that we had to add cols: declination (not dec), and the distances for each of the cosmologies

temp_and_3d_xmatch = run_3d_spatial_crossmatch(temporal_xmatch, gracedb_events)

In [9]:
temp_and_3d_xmatch

""


## Then, we'll scale up
- MOCs: we'd talked about having 300 per year
- Sources/SNs/etc: check notes, but an in-memory amount iirc